## Historical Data Compiler

**Retreive available metar data history from Aviation Weather API**:

In [1]:
# =========================
# Libraries
# =========================
import requests
import pandas as pd
from pathlib import Path

# =========================
# Configuration
# =========================
STATION = "EGLC"
CSV_FILE = f"{STATION}_metar_history.csv"

url = "https://aviationweather.gov/api/data/metar"

params = {
    "ids": STATION,
    "format": "json",
    "hours": 720
}

# =========================
# Fetch live data
# =========================
resp = requests.get(url, params=params)
resp.raise_for_status()

df_new = pd.DataFrame(resp.json())

if len(df_new) == 0:
    print("No data returned.")
    exit()

# =========================
# Time normalization
# =========================
df_new["obsTime"] = pd.to_datetime(df_new["obsTime"], unit="s", utc=True)

# =========================
# Rename to canonical schema
# =========================
df_new = df_new.rename(columns={
    "temp": "temp_c",
    "dewp": "dewpoint_c",
    "wdir": "wind_dir",
    "wspd": "wind_speed",
    "wgst": "wind_gust",
    "altim": "pressure"
})

# =========================
# Add source tag
# =========================
df_new["source"] = "aviationweather"

# =========================
# Ensure canonical columns exist
# =========================
canonical_cols = [
    "obsTime",
    "temp_c",
    "dewpoint_c",
    "wind_dir",
    "wind_speed",
    "wind_gust",
    "pressure",
    "source"
]

for col in canonical_cols:
    if col not in df_new.columns:
        df_new[col] = pd.NA

df_new = df_new[canonical_cols]

# =========================
# Load existing history
# =========================
if Path(CSV_FILE).exists():
    df_existing = pd.read_csv(CSV_FILE)

    df_existing["obsTime"] = pd.to_datetime(
        df_existing["obsTime"],
        utc=True,
        errors="coerce"
    )

    df = pd.concat([df_existing, df_new], ignore_index=True)
else:
    df = df_new

# =========================
# Deduplicate (critical)
# =========================
df = df.drop_duplicates(subset=["obsTime"], keep="last")

# =========================
# Sort chronologically
# =========================
df = df.sort_values("obsTime")

# =========================
# Save (ISO format for stability)
# =========================
df["obsTime"] = df["obsTime"].dt.strftime("%Y-%m-%dT%H:%M:%SZ")

df.to_csv(CSV_FILE, index=False)

# =========================
# Output summary
# =========================
print(f"Saved {len(df)} total observations")
print(df.tail(5))

Saved 400 total observations
                obsTime  temp_c  dewpoint_c wind_dir  wind_speed  wind_gust  \
4  2026-06-25T10:20:00Z      29          19      100          17        NaN   
3  2026-06-25T10:50:00Z      29          19      100          18        NaN   
2  2026-06-25T11:20:00Z      29          20      100          16        NaN   
1  2026-06-25T11:50:00Z      29          20      100          16        NaN   
0  2026-06-25T12:20:00Z      29          20       80          17        NaN   

   pressure           source  
4      1018  aviationweather  
3      1018  aviationweather  
2      1017  aviationweather  
1      1017  aviationweather  
0      1017  aviationweather  


**Retreive avialable historical information from Mesonet**:

In [2]:
import requests
import pandas as pd
from io import StringIO

def load_metar_history(station, start_year=2020, end_year=2026):
    url = "https://mesonet.agron.iastate.edu/cgi-bin/request/asos.py"

    params = {
        "station": station,
        "data": "tmpf,dwpf,sknt,gust,sped,mslp",
        "year1": start_year,
        "month1": 1,
        "day1": 1,
        "year2": end_year,
        "month2": 12,
        "day2": 31,
        "tz": "UTC",
        "format": "onlycomma"
    }

    r = requests.get(url, params=params, timeout=60)
    r.raise_for_status()

    df = pd.read_csv(StringIO(r.text), low_memory=False)

    # =========================
    # TIME NORMALIZATION
    # =========================
    if "valid" in df.columns:
        df["obsTime"] = pd.to_datetime(df["valid"], errors="coerce", utc=True)
        df = df.drop(columns=["valid"])
    else:
        df["obsTime"] = pd.NaT

    # =========================
    # NUMERIC CLEANING
    # =========================
    for col in ["tmpf", "dwpf", "sknt", "gust", "sped", "mslp"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # =========================
    # UNIT CONVERSION (F → C)
    # =========================
    df["temp_c"] = (df["tmpf"] - 32) * 5 / 9
    df["dewpoint_c"] = (df["dwpf"] - 32) * 5 / 9

    # =========================
    # WIND FIELDS
    # =========================
    df["wind_speed"] = df["sknt"]  # knots (keep as-is for now)
    df["wind_gust"] = df["gust"]

    # =========================
    # PRESSURE
    # =========================
    df["pressure"] = df["mslp"]

    # =========================
    # PLACEHOLDER FIELDS (not in Mesonet)
    # =========================
    df["wind_dir"] = pd.NA
    df["source"] = "mesonet"

    # =========================
    # FINAL CANONICAL SCHEMA
    # =========================
    canonical_cols = [
        "obsTime",
        "temp_c",
        "dewpoint_c",
        "wind_dir",
        "wind_speed",
        "wind_gust",
        "pressure",
        "source"
    ]

    for col in canonical_cols:
        if col not in df.columns:
            df[col] = pd.NA

    df = df[canonical_cols]

    # =========================
    # CLEAN TIME
    # =========================
    df = df.dropna(subset=["obsTime"])
    df = df.sort_values("obsTime")

    return df


# =========================
# RUN
# =========================
df_meso = load_metar_history("EGLC")
print(df_meso.head())

                    obsTime  temp_c  dewpoint_c wind_dir  wind_speed  \
0 2020-01-01 00:20:00+00:00     5.0         3.0     <NA>         4.0   
1 2020-01-01 00:50:00+00:00     5.0         3.0     <NA>         4.0   
2 2020-01-01 01:20:00+00:00     5.0         4.0     <NA>         3.0   
3 2020-01-01 01:50:00+00:00     4.0         3.0     <NA>         3.0   
4 2020-01-01 02:20:00+00:00     3.0         3.0     <NA>         6.0   

   wind_gust  pressure   source  
0        NaN       NaN  mesonet  
1        NaN       NaN  mesonet  
2        NaN       NaN  mesonet  
3        NaN       NaN  mesonet  
4        NaN       NaN  mesonet  


**Merge Dataframes into singular historical file**:

In [3]:
import pandas as pd

CSV_FILE = "EGLC_metar_history.csv"

# =========================
# LOAD AVIATIONWEATHER DATA
# =========================
df_live = pd.read_csv(CSV_FILE)

df_live["obsTime"] = pd.to_datetime(df_live["obsTime"], utc=True, errors="coerce")

# Ensure source column exists
if "source" not in df_live.columns:
    df_live["source"] = "aviationweather"

# =========================
# df_meso is assumed already created
# (from your Mesonet loader)
# =========================

# Ensure same timestamp format
df_meso["obsTime"] = pd.to_datetime(df_meso["obsTime"], utc=True, errors="coerce")

# =========================
# ALIGN COLUMNS
# =========================
all_cols = sorted(set(df_live.columns).union(set(df_meso.columns)))

for c in all_cols:
    if c not in df_live.columns:
        df_live[c] = pd.NA
    if c not in df_meso.columns:
        df_meso[c] = pd.NA

df_live = df_live[all_cols]
df_meso = df_meso[all_cols]

# =========================
# PRIORITY MERGE RULE
# =========================

# Step 1: mark source priority
df_live["_priority"] = 2   # aviationweather = HIGH priority
df_meso["_priority"] = 1   # mesonet = lower priority

# Step 2: combine
df = pd.concat([df_live, df_meso], ignore_index=True)

# Step 3: sort so aviationweather wins on duplicates
df = df.sort_values(["obsTime", "_priority"], ascending=[True, False])

# Step 4: drop duplicates (keep aviationweather)
df = df.drop_duplicates(subset=["obsTime"], keep="first")

# Step 5: cleanup
df = df.drop(columns=["_priority"])
df = df.sort_values("obsTime")

preferred_order = [
    "obsTime",
    "temp_c",
    "dewpoint_c",
    "wind_speed",
    "wind_gust",
    "wind_dir",
    "pressure",
    "source"
]

# ensure all columns exist (safe guard)
for col in preferred_order:
    if col not in df.columns:
        df[col] = pd.NA

df = df[preferred_order]

# =========================
# SAVE BACK TO CSV
# =========================
df.to_csv(CSV_FILE, index=False)

print(f"Merged dataset complete: {len(df)} rows")
print(df.tail(5))

Merged dataset complete: 113565 rows
                      obsTime  temp_c  dewpoint_c  wind_speed  wind_gust  \
395 2026-06-25 10:20:00+00:00    29.0        19.0        17.0        NaN   
396 2026-06-25 10:50:00+00:00    29.0        19.0        18.0        NaN   
397 2026-06-25 11:20:00+00:00    29.0        20.0        16.0        NaN   
398 2026-06-25 11:50:00+00:00    29.0        20.0        16.0        NaN   
399 2026-06-25 12:20:00+00:00    29.0        20.0        17.0        NaN   

    wind_dir  pressure           source  
395      100    1018.0  aviationweather  
396      100    1018.0  aviationweather  
397      100    1017.0  aviationweather  
398      100    1017.0  aviationweather  
399       80    1017.0  aviationweather  
